In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns 
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import duckdb

In [ ]:
con = duckdb.connect("health_enforcement.duckdb")
df_mart = con.execute("SELECT * FROM enriched_mart").df()

In [ ]:
df_counts = df_mart['PENALTY_TYPE'].value_counts().reset_index()
df_counts.columns = ['PENALTY_TYPE', 'count']

df_counts['percent'] = df_counts['count'] / df_counts['count'].sum() * 100

fig = px.bar(
    df_counts,
    x='PENALTY_TYPE',
    y='count',
    title='Penalty Type Distribution',
    hover_data={'percent': ':.2f'}  
)

fig.show()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

penalty_types = ["Administrative Penalty", "Citation", "Failure to Report Penalty"]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=penalty_types,
    horizontal_spacing=0.08
)

for i, ptype in enumerate(penalty_types, start=1):
    subset = df_mart[df_mart['PENALTY_TYPE'] == ptype]

    counts = (
        subset['COMPLAINT_COUNT']
        .value_counts()
        .sort_index()
    )

    fig.add_trace(
        go.Bar(
            x=counts.values,
            y=counts.index.astype(str),
            orientation='h',
            text=[f'{v} ({v/counts.sum()*100:.1f}%)' for v in counts.values],
            textposition='outside',
            showlegend=False
        ),
        row=1, col=i
    )

fig.update_layout(
    title='Complaint Count Distribution by Penalty Type',
    template='plotly_white',
    height=650,
    width=1500
)

# allow different scales for each subplot
fig.update_xaxes(matches=None, title_text='Count')

fig.update_yaxes(title_text='Complaint Count')

fig.show()

In [ ]:


penalty_types = ["Administrative Penalty", "Citation", "Failure to Report Penalty"]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=penalty_types,
    horizontal_spacing=0.12
)

for i, ptype in enumerate(penalty_types, start=1):
    subset = df_mart[df_mart['PENALTY_TYPE'] == ptype]
    counts = subset['CLASS_ASSESSED_FINAL'].value_counts().sort_values()  # ascending for horizontal

    fig.add_trace(
        go.Bar(
            x=counts.values,
            y=counts.index,
            orientation='h',
            text=[f'{v} ({v/counts.sum()*100:.1f}%)' for v in counts.values],
            textposition='outside',
            name=ptype,
            showlegend=False
        ),
        row=1, col=i
    )

fig.update_layout(
    title='Class Assessed Final by Penalty Type',
    height=500,
    width=1400,
    template='plotly_white'
)

fig.update_xaxes(title_text='Count')

fig.show()

In [ ]:
import plotly.express as px

# total cases per penalty type
totals = df_mart.groupby('PENALTY_TYPE').size()

# appealed cases per penalty type
appealed = df_mart[df_mart['APPEALED'] == 1].groupby('PENALTY_TYPE').size()

# combine
appeal_df = (
    totals.to_frame('total')
    .join(appealed.to_frame('appealed'))
    .fillna(0)
    .reset_index()
)

# appeal rate
appeal_df['appeal_rate'] = appeal_df['appealed'] / appeal_df['total'] * 100

fig = px.bar(
    appeal_df,
    x='PENALTY_TYPE',
    y='appealed',
    title='Appealed Penalties by Penalty Type',
    text=appeal_df.apply(
        lambda r: f"{int(r['appealed'])} ({r['appeal_rate']:.1f}%)",
        axis=1
    )
)

fig.update_traces(textposition='outside')

fig.update_layout(
    template='plotly_white',
    yaxis_title='Appealed Count',
    xaxis_title='Penalty Type'
)

fig.show()

In [ ]:
df_mart.columns

In [ ]:
import pandas as pd
import plotly.express as px

# Make sure the violation date is datetime
df_mart['PENALTY_ISSUE_DATE'] = pd.to_datetime(df_mart['PENALTY_ISSUE_DATE'], errors='coerce')

# Drop rows where the date is missing
df_time = df_mart.dropna(subset=['PENALTY_ISSUE_DATE']).copy()

# Extract year
df_time['Year'] = df_time['PENALTY_ISSUE_DATE'].dt.year

# Count violations per year
violations_per_year = df_time.groupby('Year').size().reset_index(name='Violation_Count')

In [ ]:
fig = px.bar(
    violations_per_year,
    x='Year',
    y='Violation_Count',
    title='Number of Violations per Year (Based on Violation From Date)',
    labels={'Violation_Count': 'Number of Violations', 'Year': 'Year'}
)

fig.update_layout(template='plotly_white')
fig.show()

In [ ]:
fig = px.bar(
    violations_per_year,
    x='Year',
    y='Violation_Count',
    title='Number of Violations per Year (Based on Appeal Received Date)',
    labels={'Violation_Count': 'Number of Violations', 'Year': 'Year'}
)

fig.update_layout(template='plotly_white')
fig.show()

In [ ]:
df_mart[['APPEALED','TOTAL_AMOUNT_INITIAL', 'TOTAL_AMOUNT_DUE_FINAL', 'TOTAL_BALANCE_DUE',
       'TOTAL_COLLECTED_AMOUNT']].describe()

In [ ]:
import plotly.express as px
import numpy as np

df_balance = df_mart[df_mart["TOTAL_BALANCE_DUE"] > 0]["TOTAL_BALANCE_DUE"]

# Compute log10 for better scaling
log_balance = np.log10(df_balance)

fig = px.histogram(
    log_balance,
    nbins=50,
    title="Total Balance Due Distribution (log scale)",
    labels={"value": "Log10(Total Balance Due)"}
)

fig.update_xaxes(
    tickvals=[1, 2, 3, 4, 5],
    ticktext=['$10', '$100', '$1K', '$10K', '$100K']
)

fig.update_yaxes(title="Frequency")
fig.update_layout(template="plotly_white")

fig.show()

In [ ]:
df_mart['FAC_TYPE_CODE'].value_counts()

In [ ]:
con.close()